# Object Tracking

The `mediatools.vision` subpackage provides a typed interface for multi-object tracking backed by `ultralytics`. Raw model output is converted immediately into immutable domain objects, so the rest of your code never touches untyped library results.

This component is optional because of its heavy dependencies. Install it with `pip install "mediatools[vision]"` or `uv sync --extra vision`.

In [ ]:
import mediatools
from mediatools.vision import ObjectTracker

#### Load Test Data
First, let's download some public video files for demonstration.

In [ ]:
from pathlib import Path

# get environment variables for testing
import pydantic_settings
class Settings(pydantic_settings.BaseSettings):
    test_zip_file_url: str
settings = Settings()

# download zip file and dump it into a temporary directory
from example_helpers import RemoteZipDownloader
zd = RemoteZipDownloader.from_url(settings.test_zip_file_url)
temp_path = zd.path

ex_vid = temp_path / "totk_builds/op_builds.mp4"
ex_vid

## The Tracker

`ObjectTracker` wraps a YOLO model and a tracker configuration. The model weights are downloaded automatically on first use.

In [ ]:
tracker = ObjectTracker(model="yolo11n.pt", tracker="bytetrack.yaml", conf=0.25)

## Tracking a Video

`track_video` lazily yields one `TrackedFrame` per video frame. Here we take the first 30 frames with `itertools.islice`.

In [ ]:
from itertools import islice

frames = tuple(islice(tracker.track_video(ex_vid), 30))
len(frames)

### `TrackedFrame` and `TrackedObject`

A `TrackedFrame` is an immutable tuple of `TrackedObject` instances. Each object carries a `track_id` that stays consistent across frames, a class, a confidence score, and a `BoundingBox`.

In [ ]:
frames[-1]

In [ ]:
for obj in frames[-1]:
    box = obj.bounding_box
    print(f"ID {obj.track_id:03d} | {obj.class_name} ({obj.confidence:.2f}) | "
          f"({box.x_min:.0f}, {box.y_min:.0f}) -> ({box.x_max:.0f}, {box.y_max:.0f})")

## Summarizing Tracks

Because the results are plain typed objects, ordinary Python is enough to analyze them. Here we count how many frames each track appears in.

In [ ]:
from collections import Counter

frames_per_track = Counter(obj.track_id for frame in frames for obj in frame)
class_per_track = {obj.track_id: obj.class_name for frame in frames for obj in frame}

[(tid, class_per_track[tid], n) for tid, n in frames_per_track.most_common()]

## Visualizing Boxes

Use `track_frame` to track frames you read yourself, such as with `cv2.VideoCapture`. Track identities persist across calls, so create one `ObjectTracker` per video stream.

In [ ]:
import cv2
import matplotlib.pyplot as plt

stream_tracker = ObjectTracker()
cap = cv2.VideoCapture(str(ex_vid))
try:
    for _ in range(30):
        success, frame = cap.read()
        if not success:
            break
        tracked = stream_tracker.track_frame(frame)
finally:
    cap.release()

annotated = frame.copy()
for obj in tracked:
    b = obj.bounding_box
    p1, p2 = (int(b.x_min), int(b.y_min)), (int(b.x_max), int(b.y_max))
    cv2.rectangle(annotated, p1, p2, (0, 255, 0), 2)
    cv2.putText(annotated, f"{obj.track_id} {obj.class_name}", p1,
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
plt.axis("off")

## Error Handling

The interface fails fast with specific exceptions, all subclasses of `TrackingDomainError`. For example, `MissingTrackingIdError` is raised if the detector output has no tracking IDs, and `TrackingConfigurationError` if the model returns a class ID that is not in its class mapping.

In [ ]:
from mediatools.vision import TrackingDomainError

try:
    for frame in tracker.track_video(ex_vid):
        pass
except TrackingDomainError as e:
    print(f"Tracking failure: {e}")

In [ ]:
zd.cleanup()